# Branch-length-aware tree signatures

Install this checkout with `python -m pip install .` before running. This example needs only `treesignal` and Python's standard library. The Jupyter frontend is installed separately. The examples are synthetic and require no OMA downloads or external C executables. See `docs/technical-report.md` for distance definitions and biological assumptions.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from math import sqrt
from treesignal import TreeSpace, TreeAnalysis, analyze_files, generate_references

newick = '((A:1,B:1):1,(C:1,D:1):1);'
gene = TreeSpace(newick)
references = TreeSpace(newick + '((A:2,B:1):1,(C:1,D:1):1);', rooted=True)
analysis = TreeAnalysis(references)
scores = analysis.compare(gene)
assert scores['wrf'] == (0.0, 1.0)
assert scores['kf'] == (0.0, 1.0)
assert abs(scores['patristic'][1] - sqrt(3 / 6)) < 1e-12
scores

Both reference trees have the same topology. Only the pendant edge for A changes. Three of the six pairwise patristic distances change by one, so their RMS discrepancy is sqrt(3/6). Both weighted split scores are one. Native handles retain each reference's lengths and can be reused.

In [ ]:
# Absolute lengths versus relative distance structure
twice_as_long = TreeSpace('((A:2,B:2):2,(C:2,D:2):2);')
one_reference = TreeAnalysis(TreeSpace(newick, rooted=True))
raw = one_reference.compare(twice_as_long)
scaled = one_reference.compare(twice_as_long, normalize=True)
assert raw['patristic'][0] > 0
assert all(abs(values[0]) < 1e-12 for values in scaled.values())
raw, scaled

## Multiple copies per species

Map each unique gene-copy label to a species. For each pair of species, the patristic score uses the minimum (or explicitly the mean) over cross-copy paths. Ordinary wRF/KF split scores do not have a unique interpretation for this mapping, so request the patristic metric only. Normalization removes each compared matrix's mean pair distance; it is a choice about rates, not an orthology inference.

In [ ]:
paralog = TreeSpace('(((A_1:0.5,A_2:0.5):0.5,B_1:1):1,(C_1:1,D_1:1):1);')
mapping = {'A_1': 'A', 'A_2': 'A', 'B_1': 'B', 'C_1': 'C', 'D_1': 'D'}
paralog_scores = analysis.compare(paralog, metrics=('patristic',),
                                  gene_to_species=mapping, aggregation='min')
assert abs(paralog_scores['patristic'][0]) < 1e-12
paralog_scores

## One inferred family tree per file

The temporary directory below models a completed tree-inference pipeline. Substitute your own family paths and reference panel after inferring gene trees from OMA or other homologous sets. The result is a tidy TSV rather than a separate program per distance.

In [ ]:
with TemporaryDirectory() as directory:
    directory = Path(directory)
    (directory / 'family_001.nwk').write_text(newick)
    (directory / 'family_002.nwk').write_text('((A:2,B:2):2,(C:2,D:2):2);')
    output = directory / 'signatures.tsv'
    rows = analyze_files(sorted(directory.glob('*.nwk')), references,
                         output=output, normalize=True)
    assert len(rows) == 2 * 2 * 3
    print(output.read_text())

## Generate reference candidates in Python

A positive seed makes the native SPR generation reproducible. Unit lengths are synthetic, not inferred divergence times or substitution lengths. Use biologically inferred reference lengths in a real length-aware analysis. Candidate panels may revisit topologies.

In [ ]:
generated = generate_references(['A', 'B', 'C', 'D'], count=5, n_spr=1,
                                seed=42, branch_length=1.0)
assert generated.ntrees == 5
generated_scores = TreeAnalysis(generated).compare(gene)
assert len(generated_scores['patristic']) == 5
generated_scores